### 1. Instalación y configuración

In [ ]:
!pip install pyspark

In [ ]:
#Importa SparkSession (punto de entrada a Spark SQL).
from pyspark.sql import SparkSession

#Importa funciones útiles (col, hour, when, round) para manipular columnas.
from pyspark.sql.functions import (
    col,
    hour,
    when,
    round
)

### 2. Creación de la sesión spark

In [ ]:
# Crea una sesión Spark llamada NYC Taxi Partitioning.
spark = SparkSession.builder \
    .appName("NYC Taxi Partitioning") \
    .getOrCreate()

### 3. Lectura y submuestreo

In [ ]:
df = spark.read.csv(
    "yellow_tripdata_2016-03.csv",  #Lee el archivo CSV de marzo 2016.
    header=True,
    inferSchema=False
)


In [ ]:
# USAR SUBMUESTRA

df = df.limit(10000) #Usa solo 10,000 registros para pruebas (submuestra).

print("Cantidad de registros:")
print(df.count())

Cantidad de registros:
10000


### 4. Exploración inicial

In [ ]:
# VER COLUMNAS
print(df.columns)

# Muestra cantidad de registros, nombres de columnas y esquema (tipos de datos).
df.printSchema()

['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'pickup_longitude', 'pickup_latitude', 'RatecodeID', 'store_and_fwd_flag', 'dropoff_longitude', 'dropoff_latitude', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount']
root
 |-- VendorID: string (nullable = true)
 |-- tpep_pickup_datetime: string (nullable = true)
 |-- tpep_dropoff_datetime: string (nullable = true)
 |-- passenger_count: string (nullable = true)
 |-- trip_distance: string (nullable = true)
 |-- pickup_longitude: string (nullable = true)
 |-- pickup_latitude: string (nullable = true)
 |-- RatecodeID: string (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- dropoff_longitude: string (nullable = true)
 |-- dropoff_latitude: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- fare_amount: string (nullable = true)
 |-- extra: string (nullable = true)
 |-- mta_tax: 

### 5. Convesion de tipos

In [ ]:
# Convierte columnas a tipos numéricos (double) o timestamp.

df_safe = df.selectExpr(
    "VendorID",

    #Usa try_cast y try_to_timestamp para evitar errores si hay valores inválidos. 
    "try_cast(passenger_count as double) as passenger_count",

    "try_cast(trip_distance as double) as trip_distance",

    "pickup_longitude",
    "pickup_latitude",

    "RatecodeID",

    "store_and_fwd_flag",

    "dropoff_longitude",
    "dropoff_latitude",

    "payment_type",

    "try_cast(fare_amount as double) as fare_amount",

    "try_cast(tip_amount as double) as tip_amount",

    "try_cast(tolls_amount as double) as tolls_amount",

    "try_cast(total_amount as double) as total_amount",

    "try_to_timestamp(tpep_pickup_datetime) as tpep_pickup_datetime",

    "try_to_timestamp(tpep_dropoff_datetime) as tpep_dropoff_datetime"
)

print("Conversión completada")

df_safe.printSchema()

Conversión completada
root
 |-- VendorID: string (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: string (nullable = true)
 |-- pickup_latitude: string (nullable = true)
 |-- RatecodeID: string (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- dropoff_longitude: string (nullable = true)
 |-- dropoff_latitude: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)



### 6. Limpieza de datos

In [ ]:
# Elimina registros con valores nulos.
# Filtra viajes con distancia y tarifa positivas y razonables (distancia < 100 millas).

df_clean = df_safe.filter(
    (col("trip_distance").isNotNull()) &
    (col("fare_amount").isNotNull()) &
    (col("passenger_count").isNotNull()) &
    (col("trip_distance") > 0) &
    (col("trip_distance") < 100) &
    (col("fare_amount") > 0) &
    (col("passenger_count") > 0)
)

print("Registros originales:")
print(df_safe.count())

print("Registros limpios:")
print(df_clean.count())

Registros originales:
10000
Registros limpios:
9949


### 7. Variables derivadas

In [ ]:
# CREAR VARIABLE pickup_hour

#Extrae la hora del día en que inició el viaje.
df_clean = df_clean.withColumn(
    "pickup_hour",
    hour(col("tpep_pickup_datetime"))
)

df_clean.select(
    "tpep_pickup_datetime",
    "pickup_hour"
).show(5)

+--------------------+-----------+
|tpep_pickup_datetime|pickup_hour|
+--------------------+-----------+
| 2016-03-01 00:00:00|          0|
| 2016-03-01 00:00:00|          0|
| 2016-03-01 00:00:00|          0|
| 2016-03-01 00:00:00|          0|
| 2016-03-01 00:00:00|          0|
+--------------------+-----------+
only showing top 5 rows


In [ ]:
# CREAR VARIABLE time_period

# Clasifica la hora en Madrugada, Mañana, Tarde, Noche.
df_clean = df_clean.withColumn(
    "time_period",
    when(
        (col("pickup_hour") >= 0) &
        (col("pickup_hour") < 6),
        "Madrugada"
    )
    .when(
        (col("pickup_hour") >= 6) &
        (col("pickup_hour") < 12),
        "Manana"
    )
    .when(
        (col("pickup_hour") >= 12) &
        (col("pickup_hour") < 18),
        "Tarde"
    )
    .otherwise("Noche")
)

df_clean.select(
    "pickup_hour",
    "time_period"
).show(10)

+-----------+-----------+
|pickup_hour|time_period|
+-----------+-----------+
|          0|  Madrugada|
|          0|  Madrugada|
|          0|  Madrugada|
|          0|  Madrugada|
|          0|  Madrugada|
|          0|  Madrugada|
|          0|  Madrugada|
|          0|  Madrugada|
|          0|  Madrugada|
|          0|  Madrugada|
+-----------+-----------+
only showing top 10 rows


In [ ]:
# CREAR VARIABLE distance_category

#Clasifica viajes en Corta (<2 mi), Media (2–10 mi), Larga (>10 mi).
df_clean = df_clean.withColumn(
    "distance_category",
    when(
        col("trip_distance") < 2,
        "Corta"
    )
    .when(
        (col("trip_distance") >= 2) &
        (col("trip_distance") < 10),
        "Media"
    )
    .otherwise("Larga")
)

df_clean.select(
    "trip_distance",
    "distance_category"
).show(10)

+-------------+-----------------+
|trip_distance|distance_category|
+-------------+-----------------+
|          2.5|            Media|
|          2.9|            Media|
|        19.98|            Larga|
|        10.78|            Larga|
|        30.43|            Larga|
|         5.92|            Media|
|         5.72|            Media|
|          6.2|            Media|
|          0.7|            Corta|
|         7.18|            Media|
+-------------+-----------------+
only showing top 10 rows


In [ ]:
# TOTAL DE REGISTROS


total = df_clean.count()

print("Total registros:")
print(total)

Total registros:
9949


### 8. Probabilidades

In [ ]:
# PROBABILIDADES DE time_period

#Calcula la probabilidad de que un viaje ocurra en cada periodo del día.
time_probs = (
    df_clean.groupBy("time_period")
    .count()
)

time_probs = time_probs.withColumn(
    "probabilidad",
    round(col("count") / total, 4)
)

time_probs.show()

+-----------+-----+------------+
|time_period|count|probabilidad|
+-----------+-----+------------+
|  Madrugada|  173|      0.0174|
|     Manana| 9776|      0.9826|
+-----------+-----+------------+



In [ ]:
# PROBABILIDADES DE distance_category

#Probabilidad de que un viaje sea corto, medio o largo.
distance_probs = (
    df_clean.groupBy("distance_category")
    .count()
)

distance_probs = distance_probs.withColumn(
    "probabilidad",
    round(col("count") / total, 4)
)

distance_probs.show()

+-----------------+-----+------------+
|distance_category|count|probabilidad|
+-----------------+-----+------------+
|            Media| 3624|      0.3643|
|            Larga|  462|      0.0464|
|            Corta| 5863|      0.5893|
+-----------------+-----+------------+



In [ ]:
# PROBABILIDADES CONJUNTAS

#Probabilidad de que un viaje pertenezca a una combinación 
partition_probs = (
    df_clean.groupBy(
        "time_period",
        "distance_category"
    )
    .count()
)

partition_probs = partition_probs.withColumn(
    "probabilidad",
    round(col("count") / total, 4)
)

partition_probs.show(truncate=False)

+-----------+-----------------+-----+------------+
|time_period|distance_category|count|probabilidad|
+-----------+-----------------+-----+------------+
|Madrugada  |Media            |87   |0.0087      |
|Madrugada  |Larga            |16   |0.0016      |
|Madrugada  |Corta            |70   |0.007       |
|Manana     |Corta            |5793 |0.5823      |
|Manana     |Media            |3537 |0.3555      |
|Manana     |Larga            |446  |0.0448      |
+-----------+-----------------+-----+------------+



### 8. Ejemplo de partición y muestreo

In [ ]:
# EJEMPLO DE PARTICIÓN

# Filtra viajes de Madrugada y Cortos.
df_tarde_corta = df_clean.filter(
    (col("time_period") == "Madrugada") &
    (col("distance_category") == "Corta")
)

print("Cantidad de registros:")

print(df_tarde_corta.count())

df_tarde_corta.show(5)

Cantidad de registros:
70
+--------+---------------+-------------+-------------------+------------------+----------+------------------+-------------------+------------------+------------+-----------+----------+------------+------------+--------------------+---------------------+-----------+-----------+-----------------+
|VendorID|passenger_count|trip_distance|   pickup_longitude|   pickup_latitude|RatecodeID|store_and_fwd_flag|  dropoff_longitude|  dropoff_latitude|payment_type|fare_amount|tip_amount|tolls_amount|total_amount|tpep_pickup_datetime|tpep_dropoff_datetime|pickup_hour|time_period|distance_category|
+--------+---------------+-------------+-------------------+------------------+----------+------------------+-------------------+------------------+------------+-----------+----------+------------+------------+--------------------+---------------------+-----------+-----------+-----------------+
|       1|            1.0|          0.7|-73.958221435546875|40.764640808105469|       

In [ ]:
# MUESTREO DE PRUEBA

sample_df = df_tarde_corta.sample(
    withReplacement=False,
    fraction=0.10,
    seed=42
)

print("Cantidad muestra:")

print(sample_df.count())

sample_df.show(5)

Cantidad muestra:
7
+--------+---------------+-------------+-------------------+------------------+----------+------------------+-------------------+------------------+------------+-----------+----------+------------+------------+--------------------+---------------------+-----------+-----------+-----------------+
|VendorID|passenger_count|trip_distance|   pickup_longitude|   pickup_latitude|RatecodeID|store_and_fwd_flag|  dropoff_longitude|  dropoff_latitude|payment_type|fare_amount|tip_amount|tolls_amount|total_amount|tpep_pickup_datetime|tpep_dropoff_datetime|pickup_hour|time_period|distance_category|
+--------+---------------+-------------+-------------------+------------------+----------+------------------+-------------------+------------------+------------+-----------+----------+------------+------------+--------------------+---------------------+-----------+-----------+-----------------+
|       2|            2.0|          0.8|-73.966423034667969|40.761650085449219|         1|  

In [ ]:
# TABLA FINAL
partition_probs.orderBy(
    "time_period",
    "distance_category"
).show(truncate=False)

+-----------+-----------------+-----+------------+
|time_period|distance_category|count|probabilidad|
+-----------+-----------------+-----+------------+
|Madrugada  |Corta            |70   |0.007       |
|Madrugada  |Larga            |16   |0.0016      |
|Madrugada  |Media            |87   |0.0087      |
|Manana     |Corta            |5793 |0.5823      |
|Manana     |Larga            |446  |0.0448      |
|Manana     |Media            |3537 |0.3555      |
+-----------+-----------------+-----+------------+



In [ ]:
# FINALIZAR SPARK
spark.stop()

print("Proceso finalizado correctamente")

Proceso finalizado correctamente
